In [12]:
import pandas as pd
df = pd.read_csv('C:/Users/Akansh/Desktop/churn-project/cleaned_churn.csv')

# One-hot encoding — categorical columns ko 0/1 columns mein todo
df_model = pd.get_dummies(df ,drop_first=True)

In [14]:
from sklearn.model_selection import train_test_split

X = df_model.drop('Churn', axis=1)
y = df_model['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [16]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Scaling — taaki convergence warning bhi fix ho aur model fair tarike se sab features treat kare
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# class_weight='balanced' — minority class (churners) ko zyada importance dega
model = LogisticRegression(max_iter=1000, class_weight='balanced')
model.fit(X_train_scaled, y_train)

predictions = model.predict(X_test_scaled)

print('Accuracy:', accuracy_score(y_test, predictions))
print(classification_report(y_test, predictions))
print('Confusion Matrix:')
print(confusion_matrix(y_test, predictions))

Accuracy: 0.7380782918149467
              precision    recall  f1-score   support

           0       0.92      0.72      0.80      1053
           1       0.49      0.81      0.61       352

    accuracy                           0.74      1405
   macro avg       0.70      0.76      0.71      1405
weighted avg       0.81      0.74      0.75      1405

Confusion Matrix:
[[753 300]
 [ 68 284]]


In [17]:
importance = pd.Series(model.coef_[0], index=X.columns)
importance.sort_values(ascending=False).head(10)


InternetService_Fiber optic       0.621428
TotalCharges                      0.605161
StreamingMovies_Yes               0.268626
StreamingTV_Yes                   0.225933
MultipleLines_Yes                 0.184139
PaymentMethod_Electronic check    0.164059
PaperlessBilling_Yes              0.114902
SeniorCitizen                     0.095790
DeviceProtection_Yes              0.019885
MultipleLines_No phone service    0.015690
dtype: float64

In [21]:
# 1. Scale train aur test
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

# 2. Train
model = LogisticRegression(max_iter=1000, class_weight='balanced')
model.fit(X_train_scaled, y_train)

# 3. Evaluate
predictions = model.predict(X_test_scaled)
print('Accuracy:', accuracy_score(y_test, predictions))
print(classification_report(y_test, predictions))

# 4. Full dataset pe probability (deployment/CSV export ke liye)
X_scaled = pd.DataFrame(scaler.transform(X), columns=X.columns, index=X.index)
df_model['ChurnProbability'] = model.predict_proba(X_scaled)[:, 1]
df_model.to_csv('C:/Users/Akansh/Desktop/churn-project/churn_predictions.csv', index=False)


Accuracy: 0.7380782918149467
              precision    recall  f1-score   support

           0       0.92      0.72      0.80      1053
           1       0.49      0.81      0.61       352

    accuracy                           0.74      1405
   macro avg       0.70      0.76      0.71      1405
weighted avg       0.81      0.74      0.75      1405

